# EVA-02 Base 448 — DIMER image classification tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/eva02-classification-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/eva02-classification-pipeline/blob/main/tutorials/eva02_classification_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-timm%2Feva02__base__patch14__448-ffcc4d?style=flat)](https://huggingface.co/timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k)
[![Upstream](https://img.shields.io/badge/Upstream-baaivision%2FEVA-181717?style=flat&logo=github&logoColor=white)](https://github.com/baaivision/EVA)
[![arXiv](https://img.shields.io/badge/arXiv-2303.11331-b31b1b.svg)](https://arxiv.org/abs/2303.11331)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** ImageNet-1k single-label image classification (1000 classes) using the pinned EVA-02 Base patch-14 448 px weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`EVA02ClassificationPipeline`) rather than reimplementing model inference. At inference the pipeline **squash-resizes every input to a fixed 448 x 448** (aspect ratio is not preserved — a non-square image is stretched, not cropped), normalises with the CLIP mean/std from the snapshot config, runs one forward pass of the vision transformer, and applies a softmax over 1000 logits. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the architecture, weights, preprocessing config, and label space; what this repository adds is manifest verification, input validation, a fixed output contract, and the `top_k_accuracy` helper.

**Learning objectives:** bootstrap the repository in a fresh runtime, generate a synthetic default input (or upload your own), surface the pipeline's ceilings, stage and digest-verify the immutable upstream snapshot, classify through the public API, read the argmax decision and rank-ordered top-k scores correctly, compute `top_k_accuracy` only when a ground-truth class index is supplied, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** object detection, segmentation, multi-label tagging, OCR, open-vocabulary classification, feature/embedding extraction (the DINOv2 sibling covers that), or any label outside the fixed 1000 ImageNet-1k classes. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. This is the heaviest of the DIMER timm classifiers (107 GMACs per 448 px image): on the model card's GPU (RTX 5070 Ti) the verified snapshot loaded in 6.4 s and one prediction took 0.8 s; **CPU works but is slow** — the card records no CPU figure and expects it to be tens of times slower than the 224 px siblings, so allow minutes, not seconds, for the single default prediction on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 348 MB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and PIL image handling; what a softmax over class logits is and why it is not a calibrated probability.
- **Data:** the default sample is a synthetic image generated in code; BYOD is one image file, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded images remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Git clone, the pinned wheel installs, and the fetch of the missing snapshot file from the Hugging Face Hub at the immutable revision. No credentials are needed.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `torchvision`, `timm`, `huggingface-hub`, `safetensors`, `numpy`, `pillow`) are directly pinned by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on CUDA when available, otherwise on CPU; no half precision, compilation, or quantization is applied.

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/eva02-classification-pipeline.git'
REPO_NAME = 'eva02-classification-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, PIL, numpy, timm, torch
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'numpy': numpy.__version__, 'pillow': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a deterministic 320 x 240 RGB gradient built in code (red ramps left to right, green top to bottom, blue is their mean), so it needs no download, contains no personal data, and its pixel SHA-256 is printed and exported for the record; no randomness is involved, so no seed is needed. It is deliberately **not square** so that the next sections can show the squash to 448 x 448. A gradient is not a photograph of any ImageNet class, so it has **no ground truth**: whatever label the model returns is smoke/sanity evidence that the code path works, never a measure of accuracy and never benchmark evidence.

BYOD is optional and disabled by default. Expected BYOD input: one image file that Pillow can open (PNG, JPEG, WebP, ...), any mode (converted to RGB), with both sides between 1 and `MAX_IMAGE_SIDE` = 4096 px; it will be squash-resized to 448 x 448 regardless of its aspect ratio. If you know the image's ImageNet-1k class, set `GROUND_TRUTH_INDEX` to that class index (0-999; the label table is printed in Section 4 after the model loads) and Section 5 will score it; leave it at `-1` when the class is unknown. The upload stays inside this runtime.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
GROUND_TRUTH_INDEX = -1  # @param {type:"integer"}
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    sample_kind = 'BYOD upload'
else:
    # Deterministic synthetic gradient: no randomness, so no seed is needed and the digest is stable.
    width, height = 320, 240
    red = np.tile(np.linspace(0.0, 255.0, width), (height, 1))
    green = np.tile(np.linspace(0.0, 255.0, height)[:, None], (1, width))
    blue = (red + green) / 2.0
    image = Image.fromarray(np.rint(np.stack([red, green, blue], axis=-1)).astype(np.uint8))
    image_name = 'synthetic_gradient_320x240'
    sample_kind = 'synthetic (generated in this cell)'
sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample': image_name, 'sample_kind': sample_kind, 'mode': image.mode, 'size': image.size, 'ground_truth_index': None if GROUND_TRUTH_INDEX == -1 else GROUND_TRUTH_INDEX, 'pixel_sha256': sample_sha256})

## 3. Validate the input against the pipeline ceilings

The pipeline enforces three operational ceilings, imported here from the package so the values shown are the ones in force: `NUM_CLASSES` (the fixed label space, 1000), `MAX_IMAGE_SIDE` (either side, px) and `MAX_BATCH` (images per `predict` call). This cell surfaces them and checks the input and the optional ground-truth index before any model work, naming the failing condition and the corrective action; `predict()` re-applies the same rules authoritatively and raises `TypeError`/`ValueError` on its own. **What the pipeline changes about your image:** it converts to RGB and squash-resizes to exactly 448 x 448 (`crop_mode: "squash"`, `crop_pct: 1.0` in the snapshot config) — nothing is cropped or dropped, but a non-square image is distorted; the aspect ratio of the sample is printed so you can see how much. The notebook itself does not resize, crop, or subsample.

In [ ]:
from eva02_classification_pipeline import MAX_BATCH, MAX_IMAGE_SIDE, NUM_CLASSES

ceilings = {'NUM_CLASSES': NUM_CLASSES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH}
print(ceilings)
width, height = image.size
problems = []
if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
    problems.append(f'image side {image.size} outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: resize the image and rerun Section 2')
if GROUND_TRUTH_INDEX != -1 and not 0 <= GROUND_TRUTH_INDEX < NUM_CLASSES:
    problems.append(f'GROUND_TRUTH_INDEX {GROUND_TRUTH_INDEX} is not -1 or an ImageNet-1k index in 0..{NUM_CLASSES - 1}: fix the form value in Section 2')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
ground_truth = None if GROUND_TRUTH_INDEX == -1 else GROUND_TRUTH_INDEX
print({'width': width, 'height': height, 'aspect_ratio': round(max(width, height) / min(width, height), 3), 'model_input': '448x448 squash-resize (aspect ratio not preserved)', 'batch_size': 1, 'within_ceilings': True})

## 4. Stage, verify, and resolve the pinned model

The public API pins the exact upstream model repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here); timm executes no remote code and the Hub is used only at that revision. The repository commits the DIMER snapshot manifest (`weights/eva02-base-448/dimer-base-manifest.json`: model id, revision, and the byte size and SHA-256 of each snapshot file) but git-ignores the 348 MB `model.safetensors`, so a fresh clone must stage the missing file first. The package's `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, into the repository's weights directory, and returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` when everything is already staged); it refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot()` then re-hashes every listed file and raises on the first size or digest mismatch, and only afterwards does `from_pretrained` build the model from that verified directory (`source: local-snapshot`). The effective model identity, the number of labels, and the selected device are printed before inference; a few example labels show the fixed ImageNet-1k label space you would pick `GROUND_TRUTH_INDEX` from.

In [ ]:
from eva02_classification_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, EVA02ClassificationPipeline, stage_missing_files, top_k_accuracy, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'num_classes': NUM_CLASSES})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': fetched, 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot_info = verify_snapshot(WEIGHTS_DIR)
print({'snapshot_path': snapshot_info['path'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')})
pipe = EVA02ClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'source': pipe.source, 'labels': len(pipe.labels), 'precision': 'float32'})
print({'example_labels': {index: pipe.labels[index] for index in (0, 207, 782, 999)}})

## 5. Classify and evaluate when ground truth exists

`predict` returns, per image, `predicted_index`/`predicted_label` and a `top_k` list of `{label, index, score}` entries **ordered by descending score** — rank position is the class ordering, and the exported files preserve it. The decision rule is `argmax` over the 1000 softmax scores (`decision_rule` in the result); the pipeline ships no acceptance threshold, so a deployment that needs an abstain option must choose its own score cut-off on its own labelled data. `score` is a softmax output over the fixed label space, **not a calibrated probability** of being correct — an unrelated image still receives a top-1 label (the card's smoke run labelled a gradient `screen, CRT screen` at score 0.013).

**Evaluation:** the only metric the repository ships is `top_k_accuracy(predictions, targets, k)` — the fraction of images whose target index appears among the first `k` ranked indices, a discrete correctness measure that says nothing about calibration or per-class behaviour. It is computed only when a ground-truth class index was supplied, at `k=1` and `k=5`, over the single demonstrated image — a single-image tutorial figure (0.0 or 1.0) with no dispersion estimate, not an accuracy estimate for any domain. The ground-truth index must be one of the 1000 ImageNet-1k classes the checkpoint was trained on (Section 3 checks the range); classes outside that label space cannot be scored. On the synthetic default sample no metric exists and none is reported; a trivial baseline (the majority class) has no meaning for a single unlabelled image, so none is reported either. The upstream 88.7 % top-1 on the ImageNet-1k validation set is an upstream claim quoted by the model card, not something this notebook measures. The runtime figure printed below is measured on the runtime identified in Section 1 for this one image and includes the first-call warm-up.

In [ ]:
import time

started = time.perf_counter()
result = pipe.predict(image, top_k=5)
elapsed = time.perf_counter() - started
prediction = result['predictions'][0]
print({'decision_rule': result['decision_rule'], 'predicted_index': prediction['predicted_index'], 'predicted_label': prediction['predicted_label'], 'device': result['device'], 'source': result['source'], 'seconds': round(elapsed, 3)})
for rank, item in enumerate(prediction['top_k'], start=1):
    print(f"{rank:>2}. index {item['index']:>4}  score {item['score']:.4f}  {item['label']}")
metrics = {}
if ground_truth is not None:
    metrics['top_k_accuracy'] = {
        'k=1': top_k_accuracy(result['predictions'], [ground_truth], k=1),
        'k=5': top_k_accuracy(result['predictions'], [ground_truth], k=5),
    }
    print({'ground_truth_index': ground_truth, 'ground_truth_label': pipe.labels[ground_truth], 'sample_metrics': metrics, 'estimation': 'single image; tutorial evidence only'})
else:
    print('no metric is reported: no ground-truth class index was supplied, so top_k_accuracy is not computed; the prediction above is sanity evidence only')

## 6. Export outputs and provenance

Two files are written under `outputs/`: a JSON record with the full prediction (argmax decision and the rank-ordered top-k scores), the metric block (empty when no ground truth was supplied), the sample identity (name, kind, size, pixel digest, ground-truth index), the ceilings in force, the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, `torch`, `timm`, NumPy, Pillow, device, precision); and a CSV of the rank-ordered top-k table keyed by image name so every row maps back to its input. No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv
import json

os.makedirs('outputs', exist_ok=True)
payload = {
    'prediction': result,
    'metrics': metrics,
    'sample': {
        'name': image_name,
        'kind': sample_kind,
        'width': image.width,
        'height': image.height,
        'pixel_sha256': sample_sha256,
        'ground_truth_index': ground_truth,
        'model_input': '448x448 squash-resize',
    },
    'ceilings': ceilings,
    'seconds': round(elapsed, 3),
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': {'path': snapshot_info['path'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'timm': timm.__version__,
        'numpy': numpy.__version__,
        'pillow': PIL.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/eva02_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/eva02_classification_top_k.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'index', 'label', 'score'])
    for rank, item in enumerate(prediction['top_k'], start=1):
        writer.writerow([image_name, rank, item['index'], item['label'], f"{item['score']:.6f}"])
print(sorted(os.listdir('outputs')))
print('outputs/eva02_classification_result.json')

## Interpretation and limits

The predicted label is the argmax of a softmax over the fixed 1000-class ImageNet-1k label space; the `score` values are uncalibrated softmax outputs, not probabilities of correctness, and the pipeline ships no threshold. On the synthetic gradient the label is meaningless by construction and no accuracy is measured; a `top_k_accuracy` value shown for a single BYOD image is 0 or 1 and says nothing about the error rate on a domain. Every input is squash-resized to 448 x 448, so strongly non-square subjects are distorted before classification; the pipeline does not detect out-of-distribution inputs (drawings, scans, satellite tiles), blur, or capture-device drift, and it exposes no features, no detection, and no labels beyond the fixed 1000. Inference is deterministic given the same weights, device and library versions (no sampling; dropout disabled); small numeric differences between CPU and CUDA kernels can reorder near-tied classes, so results on fixed hardware are repeatable but not guaranteed bitwise-identical across devices.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated input against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, reproduction of the upstream accuracy, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/eva02-base-448/` and rerun Section 4. A `ValueError` naming `MAX_IMAGE_SIDE` or `GROUND_TRUTH_INDEX` in Section 3: fix the form values in Section 2 and rerun from there. A very slow Section 5 on a CPU runtime is expected for this 107-GMAC model; switch to a CUDA runtime or use the MobileNetV4 sibling for CPU latency.

**Next experiments.** Upload a photograph of an ImageNet-1k object with `USE_BYOD` enabled and set `GROUND_TRUTH_INDEX` from the label table to see `top_k_accuracy` at `k=1` and `k=5`; upload a strongly non-square version of the same photograph to observe the effect of the squash; compare the CUDA and CPU top-5 orderings on the same image to observe kernel-level variability. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k
- Upstream code: https://github.com/baaivision/EVA
- EVA-02 paper: https://arxiv.org/abs/2303.11331
- timm documentation: https://huggingface.co/docs/timm